## 0. Project Overview: Iron Mine Patch Acquisition

This notebook automates the acquisition of Sentinel-2 satellite imagery patches for iron mine monitoring. It utilizes the **GEEPatch** pipeline to ensure spatial alignment and radiometric consistency, which is critical for downstream deep learning tasks.

## 1. Environment Setup and Initialization

In this section, we configure the system environment, append necessary package paths, and initialize the **Google Earth Engine (GEE)** API. Authentication is handled automatically if a valid session is not detected.

In [1]:
import os
import sys
import json
import ee
import geopandas as gpd
#from tqdm.auto import tqdm
from tqdm import tqdm
from datetime import datetime

# Configure package paths and initialize GEE
print(os.getcwd())
sys.path.append(os.path.abspath('../../'))
from gee_downloader import GEEPatch

# 1. Load Package
try:
    from gee_downloader import GEEPatch
    print("GEEPatch package loaded successfully!")
except ImportError:
    print("Package not found. Please check your current directory.")
# 2. Initialize Earth Engine (Required before defining Geometry)
try:
    ee.Initialize()
    print("Google Earth Engine initialized!")
except Exception:
    print("Authentication required. Triggering auth...")
    ee.Authenticate()
    ee.Initialize()
    print("Google Earth Engine initialized via Auth!")

/raven/u/yhsuh/coding/patch_fetcher/examples/02_steel_mine
GEEPatch package loaded successfully!
Authentication required. Triggering auth...


Enter verification code:  4/1AfrIepAkrhKTi8VDwU9lkXBtnT8gC0H4_7tGiFMYnHqPKGZ9hxRo9QELn_U



Successfully saved authorization token.
Google Earth Engine initialized via Auth!


## 2. Loading Regions of Interest (ROI)

We load the target site coordinates from a pre-defined vector file. These sites represent specific iron mine locations with a 5km buffer, providing the spatial context for our data acquisition.

## 3. Preprocessing and Configuration

Before starting the download, we perform two critical steps:

1. **Geometry Optimization:** Simplifies complex polygons to improve GEE API response times and prevents timeout errors.
2. **Parameter Definition:** Sets the temporal range, cloud cover constraints, and zoom levels. We use **Zoom Level 14** (approx. 9.55 m/pixel) to match the native resolution of Sentinel-2's RGB bands.

In [8]:
# ====================================================
# [Configuration]
# ====================================================
# 1. Local file path configuration and loading
local_file_path = f"./data/ROI/"
target_crs = "EPSG:4326"
# TODO: Remove .head() for production or full dataset processing
#gdf_targets = gpd.read_file(local_file_path)#.head(n=2)

from pathlib import Path
import pandas as pd

gdfs = []

for zip_path in Path(local_file_path).glob("*.zip"):
    gdf = gpd.read_file(f"zip://{zip_path}")
    
    if gdf.crs is None:
        print(f"{zip_path} has no CRS")
        continue
        
    gdf = gdf.to_crs(target_crs)
    gdfs.append(gdf)

gdf_targets = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=target_crs)


# 2. Reproject CRS and simplify geometries
print("[Status] Executing geometry simplification...")
gdf_targets = gdf_targets.to_crs(target_crs)
gdf_targets['geometry'] = gdf_targets.geometry.simplify(tolerance=0.0001)

# 3. Parameter Configuration
START_DATE = '2022-01-01'
END_DATE = '2025-12-31'
CLOUD_THRESHOLD = 5 
ZOOM_LEVEL = 14
BANDS = ['B4', 'B3', 'B2']

# Base output directory (Handles $USER environment variable for multi-user support)
user_name = os.environ.get('USER', 'default_user')
#BASE_OUTPUT_DIR = f"/ptmp/{user_name}/iron_mines/AOI_subset"
BASE_OUTPUT_DIR = f"./out/iron_mines/AOI_subset"
print(BASE_OUTPUT_DIR)

[Status] Executing geometry simplification...
./out/iron_mines/AOI_subset


In [18]:
print(gdf_targets)

   id                                           geometry
0   1  POLYGON ((127.30957 39.52099, 127.30957 39.588...
1   1  POLYGON ((129.78633 41.78595, 129.78705 41.844...
2   1  POLYGON ((127.55127 39.09596, 127.55127 39.147...


## 4. Hybrid Data Acquisition Pipeline

This is the main execution block. To maximize throughput while respecting GEE's rate limits, the pipeline employs a **hybrid strategy**:

* **Inter-region Sequential:** Iterates through each target site one by one.
* **Intra-region Parallel:** For a specific site, patches within a scene are downloaded concurrently using multi-threading.

> **Note:** The progress bar displays the current **Scene** (unique acquisition date) being processed, rather than individual files, to provide a clearer overview of the temporal progress.

---

## 5. Data Storage Hierarchy and Naming Convention

To ensure systematic management of the multi-temporal dataset, the **GEEPatch** pipeline organizes the output files into a hierarchical directory structure based on spatial indices and meteorological seasons.

### Directory Structure

The files are stored according to the following hierarchy:

* **Root Directory:** `${BASE_OUTPUT_DIR}`
* **Level 1 (Spatial Index):** Subdirectories named by the `target_index` (e.g., `0/`, `1/`, ...).
* **Level 2 (Temporal/Seasonal):** Further categorized into four meteorological seasons: `Spring/`, `Summer/`, `Autumn/`, and `Winter/`.

### File Naming Format

Each patch is saved with a deterministic filename to facilitate automated parsing during the model training phase:

> **Format:** `image_c[Class]_[Index]_[Date]_[PatchIndex].png`

| Component | Description |
| --- | --- |
| **`Class`** | Target class label (e.g., `Mine`) |
| **`Index`** | The unique identifier of the ROI |
| **`Date`** | Acquisition date in `YYYYMMdd` format |
| **`PatchIndex`** | The specific tile identifier within the scene |

---

## 6. Rationale for Utilizing 8-bit PNG Format

The pipeline exports satellite data as **8-bit PNG (Portable Network Graphics)** files rather than traditional 16-bit GeoTIFFs. This decision is based on three critical technical requirements:

### 1. Compatibility with Deep Learning Backbones

Modern computer vision models (e.g., ResNet, ViT) are natively designed to process 8-bit RGB inputs. By performing radiometric normalization and 8-bit conversion during the acquisition phase, we eliminate the computational overhead of on-the-fly normalization during model training.

### 2. Lossless Data Integrity

Unlike JPEG, which introduces compression artifacts that can degrade spectral signals, PNG uses the **lossless DEFLATE algorithm**. This ensures that every pixel value—after the 8-bit conversion—is preserved with 100% integrity, which is vital for precise land-cover classification.

### 3. I/O Efficiency on HPC Systems

PNG files provide a significant reduction in file size while maintaining identical data quality within the 8-bit space. This efficiency reduces the I/O bottleneck when loading thousands of patches from parallel file systems during large-scale training.


In [19]:
# ====================================================
# [Helper] Season classification function
# ====================================================
def get_season(date_str: str) -> str:
    """
    Classifies the season based on the month of the provided date string (YYYYMMDD).
    
    Args:
        date_str (str): The acquisition date of the image.
        
    Returns:
        str: The corresponding meteorological season ('Spring', 'Summer', 'Autumn', 'Winter').
    """
    from datetime import datetime
    m = datetime.strptime(date_str, "%Y%m%d").month
    if m in [3, 4, 5]: return "Spring"
    elif m in [6, 7, 8]: return "Summer"
    elif m in [9, 10, 11]: return "Autumn"
    else: return "Winter"

# ====================================================
# [Main Execution] Serial processing of regions with concurrent patch downloading
# ====================================================

import os
import json
import ee
import geopandas as gpd
from tqdm import tqdm

# Initialize the downloader instance
downloader = GEEPatch()

# System Start Logs
print(f"Starting the download job...")
print(f"- Number of Target Polygons: {len(gdf_targets)}")
print(f"- Date Range: {START_DATE} to {END_DATE}")
print(f"- Max Cloud Coverage: {CLOUD_THRESHOLD}%")

# 1. Initialize Overall Progress Bar (Outer Loop)
pbar_polygons = tqdm(gdf_targets.iterrows(), total=len(gdf_targets), desc="Overall Progress")

for idx, row in pbar_polygons:
    
    # 1. Set Target Index and Class Label
    target_index = row['index'] if 'index' in row else idx
    class_label = "X" 
    
    # Update progress bar description with the current region index
    pbar_polygons.set_description(f"Processing Region: {target_index}")
    
    # 2. Geometry Conversion (GeoPandas -> Earth Engine)
    try:
        geom_json = json.loads(gpd.GeoSeries([row.geometry]).to_json())
        roi_ee = ee.Geometry(geom_json['features'][0]['geometry'])
        roi_gs = gpd.GeoSeries([row.geometry], crs="EPSG:4326")
    except Exception as e:
        print(f"Geometry conversion failed (Index: {target_index}): {e}")
        continue

    # 3. Filter Sentinel-2 Image Collection
    collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterBounds(roi_ee) \
        .filterDate(START_DATE, END_DATE) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_THRESHOLD)) \
        .select(BANDS)

    # 4. Retrieve Image Metadata List
    try:
        img_info_list = collection.map(lambda img: img.set({
            'date': img.date().format('YYYYMMdd'),
            'id': img.id()
        })).reduceColumns(ee.Reducer.toList(2), ['id', 'date']).get('list').getInfo()
    except Exception as e:
        print(f"Failed to retrieve image list for Index {target_index}: {e}")
        continue

    # Skip if no images match the criteria
    if not img_info_list:
        continue

    # 5. Image Acquisition Loop (Hybrid Processing)
    total_scenes = len(img_info_list)
    
    for i, (img_id, date_str) in enumerate(img_info_list):
        # Determine the season for the current image
        season = get_season(date_str)
        
        # [Status Update] specific image progress within the current region
        pbar_polygons.set_description(f"Processing Region {target_index} | Scene {i+1}/{total_scenes}")
        
        # Define save directory with seasonal subdirectories
        index_dir = os.path.join(BASE_OUTPUT_DIR, str(target_index))
        save_dir = os.path.join(index_dir, season)
        
        full_image_id = f"COPERNICUS/S2_SR_HARMONIZED/{img_id}"
        file_suffix = f"image_c{class_label}_{target_index}_{date_str}"

        # Execute tile download and processing
        downloader.download_as_wmts_tiles(
            image=ee.Image(full_image_id),
            roi=roi_gs, 
            output_dir=save_dir,
            zoom=ZOOM_LEVEL,
            bands=BANDS,
            filename_prefix=file_suffix, 
            vis_params={'min': [0, 0, 0], 'max': [4095, 4095, 4095]},
            show_progress=False
        )

Google Earth Engine initialized successfully.
Starting the download job...
- Number of Target Polygons: 3
- Date Range: 2022-01-01 to 2025-12-31
- Max Cloud Coverage: 5%


Processing Region 2 | Scene 43/43: 100%|██████████| 3/3 [34:12<00:00, 684.15s/it]



## 7. Rationale for Spatial Alignment: EPSG:3857 and Zoom Level 14

The **GEEPatch** pipeline enforces strict spatial alignment by utilizing the Web Mercator grid system. This deterministic approach ensures that multi-temporal datasets are perfectly co-registered at the pixel level without requiring post-hoc image registration.

### 1. Deterministic Tiling via EPSG:3857

Standard satellite data exports often result in slight pixel shifts due to varying grid anchors across different scenes. By adopting **EPSG:3857 (Web Mercator)**, we leverage a global, fixed tiling hierarchy where tile boundaries are mathematically constant at any given zoom level.

* **Temporal Consistency**: A pixel at a specific coordinate in 2022 corresponds exactly to the same physical location in 2025.
* **Deep Learning Readiness**: This "pixel-perfect" alignment allows models to learn temporal changes (e.g., mine expansion) without noise introduced by spatial misalignment.

### 2. Resolution Matching at Zoom Level 14

Sentinel-2's RGB bands (B4, B3, B2) possess a native spatial resolution of **10m**. Selecting an appropriate Zoom Level (ZL) is critical to avoid either losing data (downsampling) or creating interpolation artifacts (upsampling). The ground resolution ($S$) in a Web Mercator projection is calculated as follows:

$$S = \frac{C \cdot \cos(\text{latitude})}{2^{\text{zoom} + 8}}$$

Where $C$ is the Earth’s equatorial circumference ($\approx 40,075,016$ meters). At the equator ($\cos(0)=1$):



* **Zoom Level 13**: $\approx 19.11$ m/pixel (Too coarse; results in information loss)
* **Zoom Level 14**: $\approx 9.55$ m/pixel (The "Sweet Spot")
* **Zoom Level 15**: $\approx 4.78$ m/pixel (Too fine; creates redundant data and increases storage).

**Zoom Level 14** provides a resolution of **9.55m**, which is the closest match to Sentinel-2’s **10m** capability. This ensures that we preserve the maximum amount of spectral information while keeping the file size and computational load optimized for computing environments.

### 3. Standards Compliance and I/O Efficiency

Using a standard Web Mercator Tile Service (WMTS) structure improves system interoperability:

* **Parallel Processing**: The grid-based nature of EPSG:3857 facilitates the "Intra-region Parallel" strategy, allowing multiple threads to fetch independent tiles simultaneously.
* **Ease of Validation**: Acquired patches can be overlaid directly on standard web maps (like Google Maps or OpenStreetMap) for rapid visual quality control.



## Satellite Data Source

Sentinel-2 MultiSpectral Instrument (MSI) Level-2A (Bottom-of-Atmosphere reflectance) data were utilized for this study. We accessed the `COPERNICUS/S2_SR_HARMONIZED` collection via the Google Earth Engine (GEE) platform. To minimize atmospheric noise and ensure clear visibility of surface features, we filtered the collection for images acquired between January 2022 and December 2025 with a granular cloud cover threshold of less than 5%.


## Spatially Aligned Patch Generation

Standard export methods often introduce sub-pixel misalignment due to inconsistent grid anchoring or resampling. To address this, we developed a deterministic tiling pipeline, named **GEEPatch**, which enforces a strict alignment to the Web Mercator (EPSG:3857) grid system.

For each Region of Interest (ROI), the pipeline calculates a fixed affine transform matrix corresponding to Zoom Level 14 (approx. 9.55 m/pixel at the equator). This ensures that a specific pixel  in a patch from date  spatially corresponds exactly to the same pixel in a patch from date , eliminating the need for post-hoc image registration. The resulting patches were generated with a dimension of  pixels.



## Radiometric Normalization and Bit-Depth Conversion

Raw reflectance values from Sentinel-2 are stored as 16-bit integers, typically scaled by 10,000. To make the data compatible with standard deep learning vision backbones (e.g., ResNet, ViT), we converted the data into 8-bit integers.

We applied a linear min-max normalization strategy to map the surface reflectance values to the  range. The normalization formula is defined as:

where  is the input reflectance. We set  and  (corresponding to 30% reflectance). While the full dynamic range of Sentinel-2 extends to 10,000, typical spectral signatures of iron mines and urban surfaces fall within the  range. This threshold was chosen to maximize contrast in the target features while preventing the quantization artifacts that would arise from compressing the full 0-10000 range into 8 bits. Values exceeding 3000 (e.g., highly reflective clouds or snow) were saturated to 255.


---